# Solving Linear Optimization Models

In the last lesson, we took a geometrical approach to looking at the solution space of a linear optimization model, where a point within the space satisfies all the constraints of the model. We also learned that an optimal solution, if it exists, is always found at a corner of the solution space, where constraints intersect. This fact is leveraged by algorithms for solving linear optimization problems.

To get a better intuition for the solution space of linear optimization models and the nature of the optimal solution, let's return to the linear optimization model we looked at in the lesson where we are maximizing profit based on the number of washers and dryers produced per day, subject to manufacturing, assembly and construction constraints. The interactive plot below (make sure to run the cell) allows the decision variables to be adjusted, showing the updated constraint values, objective function values, and location of the currrent decision point within the solution space. Keep your eye on the objective function value. How high can you get the objective function without violating any constraints? Where in the solution space do you find the highest objective value.

Clicking the "Show Objective Level Sets" checkbox plots the objective function for several fixed objective function values - in other words, these dashed lines show areas in the solution space where the objective function takes on a particular value. You can see that this increases in one direction, perpendicular to the dashed lines. This gives some intuition as to why optimal solutions are found at corners of the space: if you are not at a corner, you can move in the direction of increasing objective function value along an edge or through the feasible space and improve your objective function value (or at least, not decrease it). As usual, don't worry about the details of the animation code unless you are interested!

In [ ]:
#@title Linear Model Solution Space

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math

# Define the coefficients for the objective function and constraints
objective_coeffs = [100, 120]  # Coefficients of the objective function
constraint_coeffs = [
    [1, 2],  # Manufacturing constraint
    [2, 1],  # Assembly constraint
    [2, 2]   # Testing constraint
]
constraint_bounds = [20, 20, 25]  # Right-hand side values for constraints

# Define level sets for the objective function
level_values = np.arange(200, 2400, 400)  # Level sets from 200 to 2200 at intervals of 400
level_colors = plt.cm.viridis(np.linspace(0, 1, len(level_values)))  # Generate unique colors for each level set

# Create sliders for decision variables
x1_slider = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.5, description="x₁ (Washers)")
x2_slider = widgets.FloatSlider(value=0.0, min=0.0, max=15.0, step=0.5, description="x₂ (Dryers)")

# Create a checkbox for level sets
level_set_checkbox = widgets.Checkbox(value=False, description="Show Objective Level Sets")

# Output widget for displaying equations
output = widgets.Output()

# Hardcoded intersection points
intersection_points = [
    (0, 10),  # Intersection of Testing with y-axis
    (5, 7.5),  # Intersection of Assembly and Testing
    (7.5, 5),  # Intersection of Manufacturing and Assembly
    (10, 0),  # Intersection of Manufacturing with x-axis
    (0, 0)  # Origin
]

# Function to compute the feasible region plot
def plot_feasible_region(x1, x2, show_level_sets=False):
    # Generate plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot constraints
    x = np.linspace(0, 15, 300)
    y1 = (constraint_bounds[0] - constraint_coeffs[0][0] * x) / constraint_coeffs[0][1]  # Manufacturing
    y2 = (constraint_bounds[1] - constraint_coeffs[1][0] * x) / constraint_coeffs[1][1]  # Assembly
    y3 = (constraint_bounds[2] - constraint_coeffs[2][0] * x) / constraint_coeffs[2][1]  # Testing

    # Plot feasible region constraints
    ax.plot(x, y1, label="Manufacturing (Red)", color='red')
    ax.plot(x, y2, label="Assembly (Blue)", color='blue')
    ax.plot(x, y3, label="Testing (Green)", color='green')

    # Shade feasible region
    ax.fill_between(x, 0, np.minimum(np.minimum(y1, y2), y3), color='gray', alpha=0.2)

    # Plot the intersection points
    for point in intersection_points:
        ax.scatter(*point, color='red', s=50)
        ax.text(point[0] + 0.2, point[1] + 0.2, f"({point[0]:.1f}, {point[1]:.1f})", fontsize=10, color='black')

    # Plot the current slider point
    ax.scatter(x1, x2, color='black', s=100, label="Current Point")
    ax.text(x1 + 0.2, x2 + 0.2, f"({x1:.1f}, {x2:.1f})", fontsize=10, color='black', bbox=dict(facecolor='yellow', alpha=0.5))

    # Plot level sets if checkbox is checked
    if show_level_sets:
        for level, color in zip(level_values, level_colors):
            y_level = (level - objective_coeffs[0] * x) / objective_coeffs[1]
            ax.plot(x, y_level, linestyle='--', color=color, label=f"Z = {level}")

    # Set plot limits and labels
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 15)
    ax.set_xlabel(r"$x_1$ (Washers)")
    ax.set_ylabel(r"$x_2$ (Dryers)")
    ax.legend()
    ax.grid(True)
    plt.show()

# Function to dynamically update the equations and plot
def update(change):
    with output:
        # Clear previous output
        output.clear_output(wait=True)

        # Get current slider values
        x1 = x1_slider.value
        x2 = x2_slider.value

        # Compute the objective function value
        Z = objective_coeffs[0] * x1 + objective_coeffs[1] * x2

        # Compute constraint values
        constraint_values = [
            constraint_coeffs[0][0] * x1 + constraint_coeffs[0][1] * x2,  # Manufacturing
            constraint_coeffs[1][0] * x1 + constraint_coeffs[1][1] * x2,  # Assembly
            constraint_coeffs[2][0] * x1 + constraint_coeffs[2][1] * x2   # Testing
        ]

        # Determine constraint statuses
        statuses = [
            r"\textcolor{green}{Satisfied}" if constraint_values[0] <= constraint_bounds[0] else r"\textcolor{red}{Violated}",
            r"\textcolor{green}{Satisfied}" if constraint_values[1] <= constraint_bounds[1] else r"\textcolor{red}{Violated}",
            r"\textcolor{green}{Satisfied}" if constraint_values[2] <= constraint_bounds[2] else r"\textcolor{red}{Violated}"
        ]

        # Display the objective function
        objective_latex = rf"Maximize \,  100x_1 + 120x_2 = 100({x1:.1f}) + 120({x2:.1f}) = {Z:.1f}"
        display(Math(objective_latex))

        # Display the constraints
        for coeff, bound, value, status in zip(constraint_coeffs, constraint_bounds, constraint_values, statuses):
            constraint_latex = rf"{coeff[0]}x_1 + {coeff[1]}x_2 \leq {bound}" \
                               rf" \quad \text{{where }} {coeff[0]}({x1:.1f}) + {coeff[1]}({x2:.1f}) = {value:.1f}" \
                               rf" \quad \textbf{{{status}}}"
            display(Math(constraint_latex))

        # Add non-negativity constraints
        non_negativity_latex = r"x_1, x_2 \geq 0"
        display(Math(non_negativity_latex))

        # Plot the feasible region
        plot_feasible_region(x1, x2, show_level_sets=level_set_checkbox.value)

# Attach the update function to sliders and checkbox
x1_slider.observe(update, names='value')
x2_slider.observe(update, names='value')
level_set_checkbox.observe(update, names='value')

# Display the layout
layout = widgets.VBox([x1_slider, x2_slider, level_set_checkbox, output])
display(layout)

# Initialize the equations and plot
update(None)
